# Upload CSV to DRP (safe typing)

Separate notebook to upload enriched CSV into a new DRP table.

It includes a safe pre-processing step for mixed `object` columns to avoid `pyarrow` conversion errors.

In [ ]:
import pandas as pd
from pathlib import Path

from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Parameters
csv_path = '/home/jovyan/documents/Equaring/output/excel_3months_enriched_2026Q1.csv'

drp_schema = 'sbx_da'
drp_table = 'tmp_excel_3months_enriched_2026q1'
drp_mode = 'replace'  # replace / append

print('csv_path =', csv_path)
print('target =', f'{drp_schema}.{drp_table}')
print('mode =', drp_mode)

In [ ]:
# Read CSV as object and normalize mixed values safely for DRP write.
df = pd.read_csv(csv_path, dtype=object)

# Optional typed columns.
if 'report_month' in df.columns:
    df['report_month'] = pd.to_datetime(df['report_month'], errors='coerce')

# Normalize text-like columns and convert null markers to Python None.
for c in df.columns:
    if c == 'report_month':
        continue
    s = df[c]
    s = s.where(~s.isna(), None)
    s = s.map(lambda x: str(x).strip() if x is not None else None)
    s = s.replace({'': None, 'None': None, 'nan': None, 'NaN': None, '<NA>': None})
    df[c] = s.astype(object)

print('rows =', len(df))
print('cols =', len(df.columns))
print('dtypes preview:')
display(df.dtypes.reset_index().rename(columns={'index': 'column', 0: 'dtype'}).head(30))
display(df.head(5))

In [ ]:
# Fill your DRP credentials here and run cell.
drp_conn = connect(
    to='DRP',
    user_params={
        'user_name': 'PUT_YOUR_DRP_LOGIN_HERE',
        'password': 'PUT_YOUR_DRP_PASSWORD_HERE',
    }
)

target_fq = f'{drp_schema}.{drp_table}'

# 1) Primary attempt: direct write
try:
    with drp_conn:
        drp_conn.write(
            table=target_fq,
            df=df,
            mode=drp_mode,
        )
    print('Upload finished (direct):', target_fq)

except Exception as e:
    print('Direct write failed:', repr(e))
    print('Fallback: manual CREATE TABLE as Nullable(String) + append')

    # 2) Fallback: force all columns to plain string/object
    df_upload = df.copy()
    for c in df_upload.columns:
        df_upload[c] = df_upload[c].map(lambda x: None if pd.isna(x) else str(x))
        df_upload[c] = df_upload[c].astype(object)

    # 3) Build PostgreSQL-safe CREATE TABLE statement
    col_defs = []
    for c in df_upload.columns:
        col_name = str(c).replace('"', '""')
        col_defs.append(f'"{col_name}" TEXT')

    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {target_fq}
    (
      {', '.join(col_defs)}
    )
    """

    with drp_conn:
        if drp_mode == 'replace':
            drp_conn.execute(f'DROP TABLE IF EXISTS {target_fq}')
        drp_conn.execute(create_sql)
        drp_conn.write(
            table=target_fq,
            df=df_upload,
            mode='append',
        )

    print('Upload finished (fallback):', target_fq)

In [ ]:
# Verification query
target_fq = f'{drp_schema}.{drp_table}'

sql_check = f"""
select count(*) as rows_cnt
from {target_fq}
"""

sql_sample = f"""
select *
from {target_fq}
limit 20
"""

with drp_conn:
    cnt_df = drp_conn.fetch(sql_check)
    sample_df = drp_conn.fetch(sql_sample)

print('rows in table:')
display(cnt_df)
print('sample:')
display(sample_df)